# Boop DQN Training

Train two DQN agents via self-play to master the board game **Boop**.

**Boop rules (brief):**
- 6x6 board, 2 players
- Place kittens or cats; placing a piece boops (pushes) adjacent pieces one step away
- Kittens can't push cats; pieces pushed off the board return to their owner's hand
- 3 kittens in a row → promote to cats; 3 cats in a row → win

**Metrics printed every `EVAL_EVERY` episodes:**
- `loss` — mean TD loss across both agents
- `vs_random` — win rate of agent 0 vs a random policy
- `vs_prev` — win rate of agent 0 vs the previous checkpoint
- `ELO` — rating anchored at 600 for a random agent

In [ ]:
%pip install open_spiel -q

In [ ]:
# Boop board game -- inlined for Colab (matches open_spiel/python/games/boop.py)
# Action encoding: piece_type * 36 + row * 6 + col  (72 actions total)
# Observation tensor: 5 board planes (6x6) + 4 hand scalars = 184 floats

import numpy as np
from open_spiel.python.observation import IIGObserverForPublicInfoGame
import pyspiel

_NUM_PLAYERS = 2
_ROWS = 6
_COLS = 6
_NUM_CELLS = _ROWS * _COLS
_NUM_PIECE_TYPES = 2
_NUM_ACTIONS = _NUM_PIECE_TYPES * _NUM_CELLS  # 72
_MAX_KITTENS = 8
_MAX_CATS = 6
_MAX_GAME_LENGTH = 500

_EMPTY = 0
_P0_KITTEN = 1
_P0_CAT = 2
_P1_KITTEN = 3
_P1_CAT = 4

_KITTEN_VAL = [_P0_KITTEN, _P1_KITTEN]
_CAT_VAL = [_P0_CAT, _P1_CAT]
_PIECE_VALS = [[_P0_KITTEN, _P0_CAT], [_P1_KITTEN, _P1_CAT]]

_GAME_TYPE = pyspiel.GameType(
    short_name='python_boop',
    long_name='Python Boop',
    dynamics=pyspiel.GameType.Dynamics.SEQUENTIAL,
    chance_mode=pyspiel.GameType.ChanceMode.DETERMINISTIC,
    information=pyspiel.GameType.Information.PERFECT_INFORMATION,
    utility=pyspiel.GameType.Utility.ZERO_SUM,
    reward_model=pyspiel.GameType.RewardModel.TERMINAL,
    max_num_players=_NUM_PLAYERS,
    min_num_players=_NUM_PLAYERS,
    provides_information_state_string=True,
    provides_information_state_tensor=False,
    provides_observation_string=True,
    provides_observation_tensor=True,
    parameter_specification={})

_GAME_INFO = pyspiel.GameInfo(
    num_distinct_actions=_NUM_ACTIONS,
    max_chance_outcomes=0,
    num_players=_NUM_PLAYERS,
    min_utility=-1.0,
    max_utility=1.0,
    utility_sum=0.0,
    max_game_length=_MAX_GAME_LENGTH)


class BoopGame(pyspiel.Game):
  def __init__(self, params=None):
    super().__init__(_GAME_TYPE, _GAME_INFO, params or dict())

  def new_initial_state(self):
    return BoopState(self)

  def make_py_observer(self, iig_obs_type=None, params=None):
    if ((iig_obs_type is None) or
        (iig_obs_type.public_info and not iig_obs_type.perfect_recall)):
      return BoopObserver(params)
    return IIGObserverForPublicInfoGame(iig_obs_type, params)


class BoopState(pyspiel.State):
  def __init__(self, game):
    super().__init__(game)
    self._cur_player = 0
    self._is_terminal = False
    self._winner = None
    self._move_count = 0
    self.board = np.zeros((_ROWS, _COLS), dtype=np.int8)
    self._hand = [[_MAX_KITTENS, 0], [_MAX_KITTENS, 0]]

  def current_player(self):
    return pyspiel.PlayerId.TERMINAL if self._is_terminal else self._cur_player

  def _legal_actions(self, player):
    actions = []
    for r in range(_ROWS):
      for c in range(_COLS):
        if self.board[r, c] == _EMPTY:
          cell = r * _COLS + c
          if self._hand[player][0] > 0:
            actions.append(cell)
          if self._hand[player][1] > 0:
            actions.append(_NUM_CELLS + cell)
    return sorted(actions)

  def _apply_action(self, action):
    piece_type = action // _NUM_CELLS
    cell = action % _NUM_CELLS
    r, c = cell // _COLS, cell % _COLS
    p = self._cur_player
    self._hand[p][piece_type] -= 1
    self.board[r, c] = _PIECE_VALS[p][piece_type]
    self._boop(r, c, is_cat=(piece_type == 1))
    self._move_count += 1
    if self._move_count >= _MAX_GAME_LENGTH:
      self._is_terminal = True
      return
    for player in (p, 1 - p):
      if self._check_win(player):
        self._is_terminal = True
        self._winner = player
        return
    self._promote_kittens(p)
    self._promote_kittens(1 - p)
    for player in (p, 1 - p):
      if self._check_win(player):
        self._is_terminal = True
        self._winner = player
        return
    self._cur_player = 1 - p

  def _action_to_string(self, player, action):
    pt = action // _NUM_CELLS
    cell = action % _NUM_CELLS
    r, c = cell // _COLS, cell % _COLS
    piece = 'cat' if pt else 'kitten'
    return f'p{player}:{piece}@({r},{c})'

  def is_terminal(self):
    return self._is_terminal

  def returns(self):
    if self._winner == 0:
      return [1.0, -1.0]
    if self._winner == 1:
      return [-1.0, 1.0]
    return [0.0, 0.0]

  def __str__(self):
    syms = {
        _EMPTY: '.', _P0_KITTEN: 'k', _P0_CAT: 'K',
        _P1_KITTEN: 'o', _P1_CAT: 'O',
    }
    rows = [
        ''.join(syms[self.board[r, c]] for c in range(_COLS))
        for r in range(_ROWS)
    ]
    rows.append(
        f'P0: {self._hand[0][0]}k {self._hand[0][1]}K  '
        f'P1: {self._hand[1][0]}k {self._hand[1][1]}K  '
        f'move={self._move_count}')
    return '\n'.join(rows)

  def _boop(self, r, c, is_cat):
    for dr in (-1, 0, 1):
      for dc in (-1, 0, 1):
        if dr == 0 and dc == 0:
          continue
        nr, nc = r + dr, c + dc
        if not (0 <= nr < _ROWS and 0 <= nc < _COLS):
          continue
        neighbor = self.board[nr, nc]
        if neighbor == _EMPTY:
          continue
        neighbor_is_cat = neighbor in (_P0_CAT, _P1_CAT)
        if not is_cat and neighbor_is_cat:
          continue
        dest_r, dest_c = nr + dr, nc + dc
        owner = 0 if neighbor in (_P0_KITTEN, _P0_CAT) else 1
        n_type = 1 if neighbor_is_cat else 0
        if not (0 <= dest_r < _ROWS and 0 <= dest_c < _COLS):
          self.board[nr, nc] = _EMPTY
          self._hand[owner][n_type] += 1
        elif self.board[dest_r, dest_c] == _EMPTY:
          self.board[dest_r, dest_c] = neighbor
          self.board[nr, nc] = _EMPTY

  def _promote_kittens(self, player):
    kitten_val = _KITTEN_VAL[player]
    cat_val = _CAT_VAL[player]
    to_promote = set()
    for r in range(_ROWS):
      for c in range(_COLS):
        for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
          cells = []
          for k in range(3):
            nr, nc = r + dr * k, c + dc * k
            if (0 <= nr < _ROWS and 0 <= nc < _COLS
                and self.board[nr, nc] == kitten_val):
              cells.append((nr, nc))
            else:
              break
          if len(cells) == 3:
            to_promote.update(cells)
    if not to_promote:
      return
    n = len(to_promote)
    cats_on_board = int(np.sum(self.board == cat_val))
    for pr, pc in to_promote:
      self.board[pr, pc] = _EMPTY
      self._hand[player][0] += 1
    cats_to_add = min(n, max(0, _MAX_CATS - cats_on_board - self._hand[player][1]))
    self._hand[player][1] += cats_to_add

  def _check_win(self, player):
    cat_val = _CAT_VAL[player]
    for r in range(_ROWS):
      for c in range(_COLS):
        for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
          if all(
              0 <= r + dr * k < _ROWS
              and 0 <= c + dc * k < _COLS
              and self.board[r + dr * k, c + dc * k] == cat_val
              for k in range(3)):
            return True
    return False


class BoopObserver:
  def __init__(self, params):
    if params:
      raise ValueError(f'Observation parameters not supported; passed {params}')
    board_size = 5 * _ROWS * _COLS
    self.tensor = np.zeros(board_size + 4, np.float32)
    self.dict = {
        'observation': np.reshape(self.tensor[:board_size], (5, _ROWS, _COLS)),
        'hand': self.tensor[board_size:],
    }

  def set_from(self, state, player):
    self.tensor.fill(0)
    obs = self.dict['observation']
    hand = self.dict['hand']
    opp = 1 - player
    mk, mc = _KITTEN_VAL[player], _CAT_VAL[player]
    ok, oc = _KITTEN_VAL[opp], _CAT_VAL[opp]
    for r in range(_ROWS):
      for c in range(_COLS):
        v = state.board[r, c]
        if v == _EMPTY:   obs[0, r, c] = 1.0
        elif v == mk:     obs[1, r, c] = 1.0
        elif v == mc:     obs[2, r, c] = 1.0
        elif v == ok:     obs[3, r, c] = 1.0
        elif v == oc:     obs[4, r, c] = 1.0
    hand[0] = state._hand[player][0] / _MAX_KITTENS
    hand[1] = state._hand[player][1] / _MAX_CATS
    hand[2] = state._hand[opp][0] / _MAX_KITTENS
    hand[3] = state._hand[opp][1] / _MAX_CATS

  def string_from(self, state, player):
    del player
    return str(state)


pyspiel.register_game(_GAME_TYPE, BoopGame)

In [ ]:
game = pyspiel.load_game('python_boop')
state = game.new_initial_state()
print('Game:', game.get_type().long_name)
print('Num distinct actions:', game.num_distinct_actions())
print('Observation tensor size:', game.observation_tensor_size())
print()
print('Initial state:')
print(state)

In [ ]:
import copy
import numpy as np
import torch
from open_spiel.python import rl_environment
from open_spiel.python.pytorch import dqn as dqn_pt
from open_spiel.python.algorithms import random_agent


def eval_win_rate(agent0, agent1, game, num_episodes=200):
    '''Win rate of agent0 (as P0) vs agent1 (as P1).'''
    env = rl_environment.Environment(game)
    wins = 0
    for _ in range(num_episodes):
        time_step = env.reset()
        ep_agents = [agent0, agent1]
        while not time_step.last():
            curr = time_step.observations['current_player']
            out = ep_agents[curr].step(time_step, is_evaluation=True)
            time_step = env.step([out.action])
        if time_step.rewards[0] > 0:
            wins += 1
    return wins / num_episodes


def update_elo(elo, opp_elo, win_rate, k=32):
    expected = 1.0 / (1.0 + 10 ** ((opp_elo - elo) / 400.0))
    return elo + k * (win_rate - expected)

In [ ]:
# ---- Hyperparameters ----
NUM_EPISODES = 20_000
EVAL_EVERY = 500
EVAL_EPISODES = 200
HIDDEN_LAYERS = [128, 128]
LR = 1e-3
BATCH_SIZE = 128
REPLAY_CAPACITY = 20_000
EPSILON_START = 0.8
EPSILON_END = 0.05
EPSILON_DECAY = 15_000
UPDATE_TARGET_EVERY = 500
LEARN_EVERY = 10
RANDOM_ELO = 600.0

# ---- Environment ----
game = pyspiel.load_game('python_boop')
env = rl_environment.Environment(game)
info_state_size = game.observation_tensor_size()  # 184
num_actions = game.num_distinct_actions()         # 72

# ---- Training agents (self-play) ----
agents = [
    dqn_pt.DQN(
        player_id=i,
        state_representation_size=info_state_size,
        num_actions=num_actions,
        hidden_layers_sizes=HIDDEN_LAYERS,
        replay_buffer_capacity=REPLAY_CAPACITY,
        batch_size=BATCH_SIZE,
        learning_rate=LR,
        update_target_network_every=UPDATE_TARGET_EVERY,
        learn_every=LEARN_EVERY,
        discount_factor=0.99,
        min_buffer_size_to_learn=BATCH_SIZE,
        epsilon_start=EPSILON_START,
        epsilon_end=EPSILON_END,
        epsilon_decay_duration=EPSILON_DECAY,
        optimizer_str='adam',
    )
    for i in range(2)
]

# ---- Snapshot agents (greedy, for vs-previous evaluation) ----
snap_agents = [
    dqn_pt.DQN(
        player_id=i,
        state_representation_size=info_state_size,
        num_actions=num_actions,
        hidden_layers_sizes=HIDDEN_LAYERS,
        replay_buffer_capacity=BATCH_SIZE,
        batch_size=BATCH_SIZE,
        min_buffer_size_to_learn=BATCH_SIZE,
        epsilon_start=0.0,
        epsilon_end=0.0,
        epsilon_decay_duration=1,
        optimizer_str='adam',
    )
    for i in range(2)
]

# ---- Random baseline ----
rand_agent = random_agent.RandomAgent(player_id=1, num_actions=num_actions)

# ---- Tracking ----
elo = 1000.0
history = {'ep': [], 'loss': [], 'vs_random': [], 'vs_prev': [], 'elo': []}

print(f'State size: {info_state_size}, Actions: {num_actions}')
print(f'Training {NUM_EPISODES} episodes, evaluating every {EVAL_EVERY}')
print('-' * 70)

for ep in range(1, NUM_EPISODES + 1):
    # Self-play episode
    time_step = env.reset()
    while not time_step.last():
        curr = time_step.observations['current_player']
        out = agents[curr].step(time_step)
        time_step = env.step([out.action])
    for ag in agents:
        ag.step(time_step)

    if ep % EVAL_EVERY == 0:
        losses = [ag.loss for ag in agents if ag.loss is not None]
        loss = sum(losses) / len(losses) if losses else float('nan')

        # Win rate vs random (current P0 vs random P1)
        wr_rand = eval_win_rate(agents[0], rand_agent, game, EVAL_EPISODES)
        elo = update_elo(elo, RANDOM_ELO, wr_rand)

        # Win rate vs previous snapshot (current P0 vs snapshot P1)
        wr_prev = eval_win_rate(agents[0], snap_agents[1], game, EVAL_EPISODES)

        # Advance snapshot to current weights
        for i in range(2):
            snap_agents[i]._q_network.load_state_dict(
                copy.deepcopy(agents[i]._q_network.state_dict()))

        history['ep'].append(ep)
        history['loss'].append(loss)
        history['vs_random'].append(wr_rand)
        history['vs_prev'].append(wr_prev)
        history['elo'].append(elo)

        print(f'Ep {ep:6d} | loss={loss:.4f} | vs_random={wr_rand:.2f}'
              f' | vs_prev={wr_prev:.2f} | ELO={elo:.0f}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Boop DQN Self-Play Training', fontsize=14)

axes[0, 0].plot(history['ep'], history['loss'])
axes[0, 0].set_title('TD Loss')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Loss')

axes[0, 1].plot(history['ep'], history['vs_random'], color='tab:orange')
axes[0, 1].axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
axes[0, 1].set_title('Win Rate vs Random')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylim(0, 1)

axes[1, 0].plot(history['ep'], history['vs_prev'], color='tab:green')
axes[1, 0].axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
axes[1, 0].set_title('Win Rate vs Previous Checkpoint')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylim(0, 1)

axes[1, 1].plot(history['ep'], history['elo'], color='tab:red')
axes[1, 1].axhline(RANDOM_ELO, color='gray', linestyle='--', linewidth=0.8,
                    label=f'Random ({RANDOM_ELO:.0f})')
axes[1, 1].set_title('ELO Rating')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].legend()

plt.tight_layout()
plt.show()